In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

True

---
# File Loader and Chunking
---

In [3]:
filename = Path("/home/tacktile/SIM_Support-ChatBot/knowledge_base/RAG_KB_bot.md")

with open(filename, "r") as f:
    markdown_text = f.read()

In [4]:
print(markdown_text)

# Simple Invoice Manager (SIM) — In App Features Documentation
# Company / Business Setup

## Definition
A feature that lets you set up and manage your business details either during first-time onboarding or later from the app settings (Primary Settings, Invoice Header & Footer).

## When & Who
- Filled by the business owner during initial app setup (onboarding) and updated anytime later from settings whenever business information changes.

## Flows / Navigation:
### A. Flow for Onboarding: 

1. Install the App you can see Onboarding Screen.
2. Upload your **company logo** and **owner signature** (can be added manually).
3. Fill in the required business details (see Field Reference below).
4. Set your preferences (country, currency, financial year, date format).
5. Tap **Save / Update**. Wait a moment for the database to update.
6. Note:- [If you want access this information inside the app]

    1. Go to Settings >> Primary Setting (Transaction No., Tax Identification Type, Country, Fi

In [5]:
headers_to_split_on = [
    ('#', 'Header_1'),
    ('##', 'Header_2'),
    ('###', 'Header_3')
]

text_splitters = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on,
                           strip_headers= False)

In [6]:
chunks = text_splitters.split_text(markdown_text)

In [7]:
len(chunks)

261

In [8]:
chunks

[Document(metadata={'Header_1': 'Simple Invoice Manager (SIM) — In App Features Documentation'}, page_content='# Simple Invoice Manager (SIM) — In App Features Documentation'),
 Document(metadata={'Header_1': 'Company / Business Setup', 'Header_2': 'Definition'}, page_content='# Company / Business Setup  \n## Definition\nA feature that lets you set up and manage your business details either during first-time onboarding or later from the app settings (Primary Settings, Invoice Header & Footer).'),
 Document(metadata={'Header_1': 'Company / Business Setup', 'Header_2': 'When & Who'}, page_content='## When & Who\n- Filled by the business owner during initial app setup (onboarding) and updated anytime later from settings whenever business information changes.'),
 Document(metadata={'Header_1': 'Company / Business Setup', 'Header_2': 'Flows / Navigation:', 'Header_3': 'A. Flow for Onboarding:'}, page_content='## Flows / Navigation:  \n### A. Flow for Onboarding:  \n1. Install the App you ca

In [9]:
chunks[0]

Document(metadata={'Header_1': 'Simple Invoice Manager (SIM) — In App Features Documentation'}, page_content='# Simple Invoice Manager (SIM) — In App Features Documentation')

---
# Embeddings and Vector Stores
---

In [12]:
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

In [13]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./chroma_db"
)

print(f"Stored: {vectorstore._collection.count()} chunks in the vectorstore")

Stored: 261 chunks in the vectorstore


# Retrievers

### MMR

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={'k':4}
)

In [15]:
query = "What is Purchase Return?"

result = retriever.invoke(query)

print(f"Query: {query}")
for i, doc in enumerate(result, 1):
    print(f"[{i}] | Metadata: {doc.metadata} | Content: {doc.page_content}\n")

Query: What is Purchase Return?
[1] | Metadata: {'Header_1': 'Purchase Return'} | Content: # Purchase Return
A purchase return is created when goods purchased from a supplier are returned. The original purchase invoice is never deleted — it records that the purchase happened. A purchase return is created instead, which connects to the original purchase invoice, records what was returned, keeps the purchase record accurate, removes returned items from inventory, and adjusts the supplier balance accordingly.

[2] | Metadata: {'Header_1': 'Purchase Return', 'Header_2': 'When and Who'} | Content: ## When and Who
- A purchase return is usually created by the business that purchased the goods, typically through the purchasing department, store manager, warehouse staff, or accounts team.

[3] | Metadata: {'Header_1': 'Sale Returns, Refunds & Credit Notes', 'Header_2': 'When & Who'} | Content: ## When & Who
- Any business that accepts returns: retailers, wholesalers, e-commerce sellers.
- Also

In [16]:
result

[Document(id='9d393211-44ad-4186-991f-4603fec6b565', metadata={'Header_1': 'Purchase Return'}, page_content='# Purchase Return\nA purchase return is created when goods purchased from a supplier are returned. The original purchase invoice is never deleted — it records that the purchase happened. A purchase return is created instead, which connects to the original purchase invoice, records what was returned, keeps the purchase record accurate, removes returned items from inventory, and adjusts the supplier balance accordingly.'),
 Document(id='0933ee86-2485-4144-a2b2-2414e364fc75', metadata={'Header_1': 'Purchase Return', 'Header_2': 'When and Who'}, page_content='## When and Who\n- A purchase return is usually created by the business that purchased the goods, typically through the purchasing department, store manager, warehouse staff, or accounts team.'),
 Document(id='4ce9730a-41b0-4f3a-a419-2c8c236e19b2', metadata={'Header_1': 'Sale Returns, Refunds & Credit Notes', 'Header_2': 'When 

### Contextual Compressor Retriever

In [31]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor,
    EmbeddingsFilter,
    DocumentCompressorPipeline
)

In [32]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

In [41]:
base_retriever = vectorstore.as_retriever(
    search_kwargs={'k':3}
)

In [45]:
query = "How to create a Purchase Return?"

base_result = base_retriever.invoke(query)

print(f"Query: {query}")
for i, doc in enumerate(base_result,1):
    print(f"[{i}] | Topic- {doc.metadata.get('topic')} |")
    print(f"Content- {doc.page_content}\n")

Query: How to create a Purchase Return?
[1] | Topic- None |
Content- # Purchase Return
A purchase return is created when goods purchased from a supplier are returned. The original purchase invoice is never deleted — it records that the purchase happened. A purchase return is created instead, which connects to the original purchase invoice, records what was returned, keeps the purchase record accurate, removes returned items from inventory, and adjusts the supplier balance accordingly.

[2] | Topic- None |
Content- ## FAQs  
**Q: Why should I not delete the original purchase invoice when goods are returned to the supplier?**
A: Deleting a purchase invoice misrepresents financial and inventory history. The original purchase happened and must remain in the records. A purchase return offsets the original purchase invoice without erasing it — this is standard accounting practice.  
**Q: How to create Purchase Return?**
A: Dashboard >> Purchase Record >> Click on the Purchase >> Make Purchas

In [46]:
compressor = LLMChainExtractor.from_llm(llm)

compressor_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [47]:
final_result = compressor_retriever.invoke(query)

print(f"Query: {query}")
for i, doc in enumerate(final_result, 1):
    print(f"[{i}] | Metadata: {doc.metadata} | Content: {doc.page_content}\n")

Query: How to create a Purchase Return?
[1] | Metadata: {'Header_1': 'Purchase Return'} | Content: A purchase return is created when goods purchased from a supplier are returned. A purchase return is created instead, which connects to the original purchase invoice, records what was returned, keeps the purchase record accurate, removes returned items from inventory, and adjusts the supplier balance accordingly.

[2] | Metadata: {'Header_1': 'Purchase Return', 'Header_2': 'FAQs'} | Content: **Q: How to create Purchase Return?**  
A: Dashboard >> Purchase Record >> Click on the Purchase >> Make Purchase Return >> Select the products that you want to return >> save

[3] | Metadata: {'Header_2': 'When and Who', 'Header_1': 'Purchase Return'} | Content: - A purchase return is usually created by the business that purchased the goods, typically through the purchasing department, store manager, warehouse staff, or accounts team.



### Embedding Based Filtering

In [51]:
embd_compressor = EmbeddingsFilter(
    embeddings=embedding,
    similarity_threshold=0.6
    )

embd_retriever = ContextualCompressionRetriever(
    base_compressor=embd_compressor,
    base_retriever=base_retriever
)

In [52]:
embd_retriever

ContextualCompressionRetriever(base_compressor=EmbeddingsFilter(embeddings=OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7790416fcd70>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7790416fd6a0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True), similarity_fn=<function cosine_similarity at 0x77904038a200>, k=20, similarity_threshold=0.6), ba

In [53]:
embd_result = embd_retriever.invoke(query)

print(f"Query: {query}")
for i, doc in enumerate(embd_result, 1):
    print(f"[{i}] | Metadata: {doc.metadata} | Content: {doc.page_content}\n")

Query: How to create a Purchase Return?
[1] | Metadata: {'Header_1': 'Purchase Return'} | Content: # Purchase Return
A purchase return is created when goods purchased from a supplier are returned. The original purchase invoice is never deleted — it records that the purchase happened. A purchase return is created instead, which connects to the original purchase invoice, records what was returned, keeps the purchase record accurate, removes returned items from inventory, and adjusts the supplier balance accordingly.

[2] | Metadata: {'Header_1': 'Purchase Return', 'Header_2': 'FAQs'} | Content: ## FAQs  
**Q: Why should I not delete the original purchase invoice when goods are returned to the supplier?**
A: Deleting a purchase invoice misrepresents financial and inventory history. The original purchase happened and must remain in the records. A purchase return offsets the original purchase invoice without erasing it — this is standard accounting practice.  
**Q: How to create Purchase Re

### DocumentCompressor Pipeline

In [58]:
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[embd_compressor, compressor]
)
pipeline_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=base_retriever
)

In [ ]:
pipeline_result = pipeline_retriever.invoke(query)
print(f"Query: {query}")
for i, doc in enumerate(pipeline_result, 1):
    print(f"[{i}] | Metadata: {doc.metadata} | Content: {doc.page_content}\n")

Query: How to create a Purchase Return?
[1] | Metadata: {'Header_1': 'Purchase Return'} | Content: A purchase return is created when goods purchased from a supplier are returned. A purchase return is created instead, which connects to the original purchase invoice, records what was returned, keeps the purchase record accurate, removes returned items from inventory, and adjusts the supplier balance accordingly.

[2] | Metadata: {'Header_2': 'FAQs', 'Header_1': 'Purchase Return'} | Content: **Q: How to create Purchase Return?**  
A: Dashboard >> Purchase Record >> Click on the Purchase >> Make Purchase Return >> Select the products that you want to return >> save

[3] | Metadata: {'Header_1': 'Purchase Return', 'Header_2': 'When and Who'} | Content: - A purchase return is usually created by the business that purchased the goods, typically through the purchasing department, store manager, warehouse staff, or accounts team.

